**We'll building a small API → Python/Pandas → PostgreSQL data pipeline. The goal is to pull data from a public API, clean and validate it, store it in a database, and make the script safe to run more than once.**

**Step 1 — Extract data from a public API**
We'll use the posts endpoint from JSONPlaceholder's API documentation. It provides sample records with fields such as id, userId, title, and body. 


In [20]:
%pip install requests pandas sqlalchemy "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\K Vaishnavi\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import requests

url = "https://jsonplaceholder.typicode.com/users"

response=requests.get(url,timeout=30)

print("Status Code:", response.status_code)
response.raise_for_status()

data=response.json()

print("Data Types:\n",type(data))
print("Number of records\n",len(data))
print("First Record:", data[0])

# What you should understand
# - requests.get() fetches data from the API.
# - response.status_code helps you check whether the request succeeded.
# - response.raise_for_status() raises an error for unsuccessful HTTP responses.
# - response.json() converts the JSON response into Python objects.
# - len(data) tells you how many records were returned.

Status Code: 200
Data Types:
 <class 'list'>
Number of records
 10
First Record: {'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


**Step 2: Convert the extracted records into a DataFrame**

In [22]:
df=pd.DataFrame(data)
df.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


**Step 3 — Flatten and inspect the data**

In [25]:
df=pd.json_normalize(data)
print("Shape:\n", df.shape)
print("Columns:\n", df.columns.tolist())

df.head()

# pandas.json_normalize is a built-in function in the Python Pandas library designed
# to flatten semi-structured, nested JSON data into a flat, tabular

Shape:
 (10, 15)
Columns:
 ['id', 'name', 'username', 'email', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat', 'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs']


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


**#Step 4 — Check data quality**

In [26]:

# Check data types and non-null counts
df.info()
print()

# Check missing values
print(df.isnull().sum())

# Check duplicate records
print("Duplicate rows:\n", df.duplicated().sum())


# Check duplicate user IDs
print("Duplicate ID's:\n", df["id"].duplicated().sum())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   10 non-null     int64 
 1   name                 10 non-null     object
 2   username             10 non-null     object
 3   email                10 non-null     object
 4   phone                10 non-null     object
 5   website              10 non-null     object
 6   address.street       10 non-null     object
 7   address.suite        10 non-null     object
 8   address.city         10 non-null     object
 9   address.zipcode      10 non-null     object
 10  address.geo.lat      10 non-null     object
 11  address.geo.lng      10 non-null     object
 12  company.name         10 non-null     object
 13  company.catchPhrase  10 non-null     object
 14  company.bs           10 non-null     object
dtypes: int64(1), object(14)
memory usage: 1.3+ KB

id           

**Step 5 — Clean and validate the API data.**

In [27]:
# 5.1 Check required columns
# For this project, every user should have an id, name, and email.

required_columns=["id","name","email"]
print(df[required_columns].isnull().sum())

#5.2 Remove duplicate records
df=df.drop_duplicates(subset=["id"])
#This keeps one record per user ID. In a real pipeline, you'd investigate unexpected duplicates as well.

#5.3 Remove records with missing required fields
df=df.dropna(subset=["id","name","email"])

#5.4 Validate the result
print("Rows after cleaning:",len(df))
print("Duplicate ID's:",df["id"].duplicated().sum())
print("Missing required values:")
print(df[required_columns].isnull().sum)

id       0
name     0
email    0
dtype: int64
Rows after cleaning: 10
Duplicate ID's: 0
Missing required values:
<bound method DataFrame.sum of       id   name  email
0  False  False  False
1  False  False  False
2  False  False  False
3  False  False  False
4  False  False  False
5  False  False  False
6  False  False  False
7  False  False  False
8  False  False  False
9  False  False  False>


**Step 6 — Save the cleaned data to a CSV file**
Before connecting to a database, let's save a copy of your cleaned data. This also gives you a checkpoint if something goes wrong later.

In [28]:
df.to_csv("Cleaned_users.csv", index=False) #to_csv() saves your DataFrame to a CSV file
print("Cleaned data saved successfully!") #prevents Pandas from adding an extra index column

Cleaned data saved successfully!


**Step 7 — Load the data into local MYSQL**

Open MYSQL workbench on your laptop and execute 
SELECT VERSION()

Create a database for your project
CREATE DATABASE api_etl_project;
USE api_etl_project;

**Step 8 — Install the Python database libraries**

1. SQLAlchemy helps Python communicate with databases.
2. PyMySQL is the driver that connects Python to MySQL.

In [29]:
%pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\K Vaishnavi\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


**Step 9 — Connect Python to MySQL**

In [30]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# create_engine — creates the object Python uses to connect to your database.
# text — lets you execute a SQL query written as a string.

#Build the connection URL:This combines your database details into one connection string.
connection_url = URL.create(
    #Define your database details
    drivername="mysql+pymysql",
    username="root",    #MySQL account you're connecting with
    password="your mysql password", #Your MySQL account password
    host="localhost",   #localhost means MySQL is running on your own computer
    port=3306,  #3306 is MySQL's default port
    database="api_etl_project" #The database you created for this project
)

#Create the SQLAlchemy engine
engine=create_engine(connection_url)

#Open a database connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION()"))
    print("Connected to MySQL:", result.scalar())



Connected to MySQL: 9.1.0


**Step 10 — Load the DataFrame into MySQL**

In [31]:
df.to_sql(
    name="users", #the destination table.
    con=engine, #the MySQL connection.
    if_exists="replace", #replaces the table if it already exists.
    index=False #prevents Pandas from writing its index as an extra column.
)

print("Data Loaded Successfully!")

Data Loaded Successfully!


**Step 11 — Verify the data**

**Option 1: Run in MySQL Workbench (recommended)**

USE api_etl_project;

select count(*) as total_users
from users

select *
from users
limit 5;

**Option 2: Run from your Python notebook**
from sqlalchemy import text

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM users")
    ).scalar()

    print("Total users:", count)

    result = conn.execute(
        text("SELECT * FROM users LIMIT 5")
    )

    for row in result:
        print(row)

**Step 12 - Make your pipeline reusable**
Currently, your code runs step by step in a notebook. A real data engineering pipeline should be easy to rerun and maintain.

We'll improve it in this order:
1. Make the pipeline rerunnable without accidentally creating duplicate records or unnecessarily replacing the table.
2. Add error handling and logging.
3. Organize the code into reusable functions.
4. Save it as a Python .py script.
5. Add a few data-quality checks.

The problem is if_exists="replace": every time you rerun the code, Pandas drops the existing table and creates it again. That can remove table constraints, indexes, or other changes you made.

Instead:
df.to_sql(
    name="users",
    con=engine,
    if_exists="append",
    index=False
)

In [36]:

# Remove duplicate users based on their ID
df = df.drop_duplicates(subset=["id"])

# Check the data before loading
print("Rows to load:", len(df))
print("Duplicate IDs:", df["id"].duplicated().sum())


Rows to load: 10
Duplicate IDs: 0


In [37]:
#Load the cleaned data into MySQL
df.to_sql(
    name="users",
    con=engine,
    if_exists="replace",
    index=False
)

print("Users table refreshed successfully!")


Users table refreshed successfully!


In [38]:
#Verify the record count
from sqlalchemy import text

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM users")
    ).scalar()

    print("Total users in MySQL:", count)


Total users in MySQL: 10


**Create reusable ETL functions**
Instead of writing all the code in separate notebook cells, we'll organize it into three functions:
1. extract_data() — fetches data from the API.
2. transform_data() — flattens JSON and cleans the data.
3. load_data() — loads the processed data into MySQL.

In [39]:
#1. extract_data() — fetches data from the API.
import requests

def extract_data():
    url= "https://jsonplaceholder.typicode.com/users" #Sends a request to the API.
    response=requests.get(url,timeout=30)
    response.raise_for_status()

    data=response.json()
    print(f"Extracted {len(data)} records")

    return data

#Test the function
data=extract_data() #Defines a reusable function.

Extracted 10 records


In [40]:
#2. transform_data() — flattens JSON and cleans the data.

import pandas as pd

def transform_data(data):
    # Convert nested JSON into a flat table
    df = pd.json_normalize(data)

    # Remove duplicate users
    df = df.drop_duplicates(subset=["id"])

    # Remove rows missing required values
    required_columns = ["id", "name", "email"]
    df = df.dropna(subset=required_columns)

    print(f"Transformed {len(df)} records")

    return df

# Test the function
df = transform_data(data)

print(df[["id", "name", "email"]].head())


Transformed 10 records
   id              name                      email
0   1     Leanne Graham          Sincere@april.biz
1   2      Ervin Howell          Shanna@melissa.tv
2   3  Clementine Bauch         Nathan@yesenia.net
3   4  Patricia Lebsack  Julianne.OConner@kory.org
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca


In [41]:
#3. load_data() — loads the processed data into MySQL.

def load_data(df, engine):
    df.to_sql(
        name="users",
        con=engine,
        if_exists="replace",
        index=False
    )

    print(f"Loaded {len(df)} records into MySQL")

# Test the function
load_data(df, engine)


Loaded 10 records into MySQL


In [42]:
#Create run_pipeline()

def run_pipeline():
    try:
        # Step 1: Extract
        data = extract_data()

        # Step 2: Transform
        df = transform_data(data)

        # Step 3: Load
        load_data(df, engine)

        print("ETL pipeline completed successfully!")

    except Exception as e:
        print("ETL pipeline failed:", e)


# Run the complete pipeline
run_pipeline()


Extracted 10 records
Transformed 10 records
Loaded 10 records into MySQL
ETL pipeline completed successfully!


In [43]:
#Verify that rerunning the pipeline doesn't change the row count

from sqlalchemy import text

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM users")
    ).scalar()

print("Total users in MySQL:", count)

Total users in MySQL: 10


In [44]:

from sqlalchemy import text

def load_data(df, engine):
    records = df.to_dict(orient="records")

    query = text("""
        INSERT INTO users (id, name, email)
        VALUES (:id, :name, :email)
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            email = VALUES(email)
    """)

    with engine.begin() as conn:
        conn.execute(query, records)

    print(f"Loaded or updated {len(records)} users")

# Test the function
load_data(df, engine)


Loaded or updated 10 users


In [45]:
load_data(df, engine)
load_data(df, engine)

Loaded or updated 10 users
Loaded or updated 10 users


In [46]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM users")
    ).scalar()

print("Total users in MySQL:", count)

Total users in MySQL: 10


In [53]:
#Add logging to your pipeline

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("etl_pipeline.log"),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("api_etl")


def extract_data():
    url = "https://jsonplaceholder.typicode.com/users"

    logger.info("Starting data extraction")

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        logger.info("Extracted %d records", len(data))
        return data

    except requests.RequestException:
        logger.exception("API extraction failed")
        raise


def transform_data(data):
    logger.info("Starting data transformation")

    try:
        df = pd.json_normalize(data)

        df = df.drop_duplicates(subset=["id"])

        required_columns = ["id", "name", "email"]
        df = df.dropna(subset=required_columns)

        logger.info("Transformation completed: %d records", len(df))
        return df

    except Exception:
        logger.exception("Data transformation failed")
        raise

from sqlalchemy import text

def load_data(df, engine):
    logger.info("Starting database load")

    query = text("""
        INSERT INTO users (id, name, email)
        VALUES (:id, :name, :email)
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            email = VALUES(email)
    """)

    try:
        records = df.to_dict(orient="records")

        with engine.begin() as conn:
            conn.execute(query, records)

        logger.info("Database load completed: %d records", len(records))

    except Exception:
        logger.exception("Database load failed")
        raise
def run_pipeline():
    logger.info("ETL pipeline started")

    try:
        data = extract_data()
        df = transform_data(data)
        load_data(df, engine)

        logger.info("ETL pipeline completed successfully")

    except Exception:
        logger.exception("ETL pipeline failed")
        raise

run_pipeline()

2026-10-13 12:57:34,330 - INFO - ETL pipeline started
2026-10-13 12:57:34,335 - INFO - Starting data extraction
2026-10-13 12:57:34,501 - INFO - Extracted 10 records
2026-10-13 12:57:34,503 - INFO - Starting data transformation
2026-10-13 12:57:34,508 - INFO - Transformation completed: 10 records
2026-10-13 12:57:34,510 - INFO - Starting database load
2026-10-13 12:57:34,510 - INFO - Database load completed: 10 records
2026-10-13 12:57:34,510 - INFO - ETL pipeline completed successfully


In [54]:
if not isinstance(data, list):
    raise ValueError("Expected API response to be a list")

if data and not all(isinstance(item, dict) for item in data):
    raise ValueError("Expected each user record to be a dictionary")

In [55]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM users")
    ).scalar()

print("Total users in MySQL:", count)

Total users in MySQL: 10
